# Fasal Bazaar Intelligence — Agricultural Commodity Arbitrage Analyst
### GPU-Accelerated Decision Intelligence for Agmarknet Mandi Price Data

**Rubric mapping:**
1. **User**: Government procurement officer / FMCG commodity buyer
2. **Bottleneck**: Comparing prices + rolling anomaly detection across millions of (market, commodity, date) rows is CPU-slow; naive price-gap flags are also economically wrong without transport cost
3. **Pipeline**: GCS/BigQuery (ingest) -> cudf.pandas + Dataproc Serverless/Spark RAPIDS (analyze) -> Streamlit/Looker (act)
4. **Output**: Transport-cost-net-of-margin ranked procurement opportunities, with estimated rupee profit per truckload
5. **Acceleration proof**: multi-scale CPU vs GPU runtime curve.

**Workflow:**
- **CPU baseline pass**: fresh runtime, skip the RAPIDS install + `%load_ext cudf.pandas` cells. Run Steps 1-6, saving benchmark results as `bench_cpu.csv`.
- **GPU pass**: `Runtime > Restart session`, run the install + load cells, then rerun Steps 1-6 identically, saving as `bench_gpu.csv`.
- Step 6's chart needs both CSVs to produce the comparison.


## Step 0 — Confirm GPU runtime
Go to `Runtime > Change runtime type > T4 GPU` before running anything below.

In [ ]:
!nvidia-smi

## Step 0b — GCP authentication
synthetic data or a local/Kaggle CSV.

In [ ]:
PROJECT_ID = "fasal-bazaar-intel

# from google.colab import auth
# auth.authenticate_user()
# print(f"Authenticated. Using project: {PROJECT_ID}")

## Step 1 — Install RAPIDS (cudf.pandas)
GPU pass only — skip entirely for the CPU baseline pass.

In [ ]:
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com -q

# Fallback if the above fails:
# !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
# !python rapidsai-csp-utils/colab/pip-install.py

## Step 2 — Load the pandas accelerator (GPU pass only)
Run BEFORE importing pandas. Zero code changes needed below this point — same functions run on
CPU or GPU depending only on whether this cell ran.

In [ ]:
%load_ext cudf.pandas

## Step 3 — Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

# Optional deps — install if missing
# !pip install xgboost scikit-learn gcsfs pandas-gbq -q

import xgboost as xgb
from sklearn.preprocessing import StandardScaler

try:
    from cuml.cluster import KMeans as GPU_KMeans
    CUML_AVAILABLE = True
except ImportError:
    from sklearn.cluster import KMeans as GPU_KMeans
    CUML_AVAILABLE = False

print("pandas version:", pd.__version__)
print("cuML available:", CUML_AVAILABLE, "(falls back to sklearn KMeans if False)")

## Step 4 — Load data
### Real data — pick one:

**Option A: Kaggle**:
```python
# !pip install kagglehub -q
# import kagglehub
# path = kagglehub.dataset_download("anishaman07/agmarknet-india-commodity-prices-oct24-aug25")
```

**Option B: data.gov.in API**:
```python
# import requests
# API_KEY = "your-data-gov-in-api-key"  # register at https://data.gov.in
# url = f"https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070?api-key={API_KEY}&format=csv&limit=10000"
```

**Option C: GCS**:
```python
# df = pd.read_csv("gs://YOUR_BUCKET/agmarknet_raw.csv")
```


In [ ]:
RAW_CSV_PATH = None       # e.g. "/content/agmarknet.csv" 
MARKET_COORDS_PATH = None  # optional separate lat/long lookup, if not already in RAW_CSV_PATH
N_SYNTHETIC_ROWS = 5_000_000 

def generate_synthetic_agmarknet(n_rows=5_000_000, seed=42):
    """Realistic-shaped fallback with lat/long baked in (jittered around state capitals) so the
    transport-cost-aware arbitrage logic and multi-scale benchmark work immediately. Swap for real
    data as soon as your Agmarknet pull is ready -- real anomalies are messier and more convincing."""
    rng = np.random.default_rng(seed)

    state_capitals = {
        "Punjab": (30.7333, 76.7794), "Haryana": (30.7333, 76.7794),
        "Uttar Pradesh": (26.8467, 80.9462), "Madhya Pradesh": (23.2599, 77.4126),
        "Maharashtra": (19.0760, 72.8777), "Rajasthan": (26.9124, 75.7873),
        "Gujarat": (23.0225, 72.5714), "Bihar": (25.5941, 85.1376),
        "West Bengal": (22.5726, 88.3639), "Karnataka": (12.9716, 77.5946),
    }
    commodities = {"Wheat": 2200, "Onion": 1500, "Potato": 1200, "Rice": 2800,
                   "Tomato": 1800, "Soybean": 4200, "Maize": 1900, "Mustard": 5200}
    varieties_raw = ["Local", "local", " Local ", "Desi", "Hybrid", "hybrid ", "Grade A", "Grade-A"]

    states = list(state_capitals.keys())
    n_markets_per_state = 25
    market_rows = []
    for s in states:
        base_lat, base_lon = state_capitals[s]
        for i in range(n_markets_per_state):
            market_rows.append((f"{s}_Market_{i}", s,
                                 base_lat + rng.normal(0, 1.2), base_lon + rng.normal(0, 1.2)))
    market_df = pd.DataFrame(market_rows, columns=["Market", "State", "Lat", "Lon"])

    dates = pd.date_range("2022-01-01", "2025-08-31", freq="D")  # ~3.5 years, closer to real scale
    n_markets = len(market_df)
    rows = rng.integers(0, n_markets, size=n_rows)
    date_idx = rng.integers(0, len(dates), size=n_rows)
    commodity_names = rng.choice(list(commodities.keys()), size=n_rows)

    base_prices = np.array([commodities[c] for c in commodity_names], dtype=float)
    state_bias = rng.normal(1.0, 0.08, size=n_rows)
    market_noise = rng.normal(1.0, 0.05, size=n_rows)
    anomaly_mask = rng.random(n_rows) < 0.01
    anomaly_mult = np.where(rng.random(n_rows) < 0.5, 0.6, 1.5)
    modal = base_prices * state_bias * market_noise
    modal = np.where(anomaly_mask, modal * anomaly_mult, modal)

    df = pd.DataFrame({
        "Market": market_df["Market"].values[rows],
        "State": market_df["State"].values[rows],
        "Lat": market_df["Lat"].values[rows],
        "Lon": market_df["Lon"].values[rows],
        "Commodity": commodity_names,
        "Variety": rng.choice(varieties_raw, size=n_rows),
        "Arrival_Date": dates[date_idx],
        "Min_Price": modal * rng.uniform(0.9, 0.97, size=n_rows),
        "Max_Price": modal * rng.uniform(1.03, 1.1, size=n_rows),
        "Modal_Price": modal,
    })
    return df

if RAW_CSV_PATH:
    df_raw = pd.read_csv(RAW_CSV_PATH)
    if MARKET_COORDS_PATH:
        coords = pd.read_csv(MARKET_COORDS_PATH)
        df_raw = df_raw.merge(coords, on="Market", how="left")
    print(f"Loaded real data: {len(df_raw):,} rows")
else:
    df_raw = generate_synthetic_agmarknet(n_rows=N_SYNTHETIC_ROWS)
    print(f"Using synthetic fallback: {len(df_raw):,} rows (swap in real data via RAW_CSV_PATH)")

df_raw.head()

## Step 5 — Cleaning pipeline
Identical code runs on plain pandas or cudf.pandas. Downcasts numeric dtypes for memory efficiency at scale.

In [ ]:
def clean_pipeline(df):
    df = df.copy()
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"], errors="coerce")

    df["Variety"] = df["Variety"].astype(str).str.strip().str.lower()
    df["Commodity"] = df["Commodity"].astype(str).str.strip().str.title()
    df["State"] = df["State"].astype(str).str.strip().str.title()
    df["Market"] = df["Market"].astype(str).str.strip()

    for col in ["Min_Price", "Max_Price", "Modal_Price"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["Modal_Price", "Arrival_Date"])
    df = df[df["Modal_Price"] > 0]

    for col in ["Min_Price", "Max_Price", "Modal_Price", "Lat", "Lon"]:
        if col in df.columns:
            df[col] = df[col].astype("float32")

    return df

t0 = time.time()
df_clean = clean_pipeline(df_raw)
clean_time = time.time() - t0
print(f"Cleaning: {len(df_clean):,} rows survived, {clean_time:.2f}s")

## Step 6 — Core analysis: state median, deviation, rolling z-score
Groupby + rolling window over potentially millions of (market, commodity, date) combinations.

In [ ]:
def analyze_pipeline(df):
    df = df.sort_values(["Commodity", "Market", "Arrival_Date"])
    state_daily_median = df.groupby(["State", "Commodity", "Arrival_Date"])["Modal_Price"].transform("median")
    df["State_Median_Price"] = state_daily_median
    df["Deviation_Pct"] = (df["Modal_Price"] - df["State_Median_Price"]) / df["State_Median_Price"] * 100

    grp = df.groupby(["Market", "Commodity"])["Modal_Price"]
    df["Rolling_Mean_7d"] = grp.transform(lambda s: s.rolling(7, min_periods=3).mean())
    df["Rolling_Std_7d"] = grp.transform(lambda s: s.rolling(7, min_periods=3).std())
    df["Z_Score"] = (df["Modal_Price"] - df["Rolling_Mean_7d"]) / df["Rolling_Std_7d"].replace(0, np.nan)
    return df

t0 = time.time()
df_analyzed = analyze_pipeline(df_clean)
analyze_time = time.time() - t0
print(f"Analysis (groupby + rolling z-score): {analyze_time:.2f}s")

total_time = clean_time + analyze_time
print(f"TOTAL PIPELINE TIME (single scale): {total_time:.2f}s")

## Step 7 — Transport-cost-aware arbitrage ranking
This is the methodology upgrade over a naive price-gap flag: a market being cheaper only counts as
a real opportunity if the price gap beats the estimated trucking cost to move goods there, plus a
minimum margin. Freight assumption: ~Rs 25-30/km for a ~10-tonne truck (Rs 0.25-0.30/km per quintal)

In [ ]:
RS_PER_KM_PER_QUINTAL = 0.28   # freight cost assumption -- clearly labeled, adjustable
MARGIN_PCT = 3.0                # minimum net margin required to call it a real opportunity
LOAD_QUINTALS = 100             # reference truckload size for the profit estimate

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1r, lon1r, lat2r, lon2r = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2r - lat1r
    dlon = lon2r - lon1r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def rank_opportunities_transport_aware(df, top_n=15, load_quintals=LOAD_QUINTALS):
    latest_date = df["Arrival_Date"].max()
    today = df[df["Arrival_Date"] == latest_date].copy()

    recent = df[df["Arrival_Date"] >= latest_date - pd.Timedelta(days=3)]
    persistence = (
        recent.assign(flag=recent["Deviation_Pct"].abs() > 8)
        .groupby(["Market", "Commodity"])["flag"].sum().rename("Persistence_Days")
    )
    today = today.merge(persistence, on=["Market", "Commodity"], how="left")
    today["Persistence_Days"] = today["Persistence_Days"].fillna(0)

    results = []
    for commodity, grp in today.groupby("Commodity"):
        if len(grp) < 2:
            continue
        cheapest = grp.loc[grp["Modal_Price"].idxmin()]
        pricier = grp[grp["Modal_Price"] > cheapest["Modal_Price"] * 1.02]
        if pricier.empty:
            continue

        dist_km = haversine_km(cheapest["Lat"], cheapest["Lon"], pricier["Lat"].values, pricier["Lon"].values)
        transport_cost = dist_km * RS_PER_KM_PER_QUINTAL
        price_gap = pricier["Modal_Price"].values - cheapest["Modal_Price"]
        net_gain = price_gap - transport_cost
        net_margin_pct = net_gain / cheapest["Modal_Price"] * 100

        best_idx = np.argmax(net_gain)
        if net_margin_pct[best_idx] > MARGIN_PCT:
            sell_row = pricier.iloc[best_idx]
            results.append({
                "Commodity": commodity,
                "Buy_Market": cheapest["Market"], "Buy_State": cheapest["State"],
                "Buy_Price": round(float(cheapest["Modal_Price"]), 2),
                "Buy_Lat": float(cheapest["Lat"]), "Buy_Lon": float(cheapest["Lon"]),
                "Sell_Market": sell_row["Market"], "Sell_State": sell_row["State"],
                "Sell_Price": round(float(sell_row["Modal_Price"]), 2),
                "Sell_Lat": float(sell_row["Lat"]), "Sell_Lon": float(sell_row["Lon"]),
                "Distance_km": round(float(dist_km[best_idx]), 1),
                "Transport_Cost_Per_Quintal": round(float(transport_cost[best_idx]), 2),
                "Net_Gain_Per_Quintal": round(float(net_gain[best_idx]), 2),
                "Net_Margin_Pct": round(float(net_margin_pct[best_idx]), 2),
                "Persistence_Days": int(cheapest["Persistence_Days"]),
                "Est_Profit_Per_Truckload": round(float(net_gain[best_idx]) * load_quintals, 0),
            })

    out = pd.DataFrame(results)
    if len(out):
        out = out.sort_values("Net_Margin_Pct", ascending=False).head(top_n)
    return out

opportunities = rank_opportunities_transport_aware(df_analyzed)
print(f"Transport-cost-aware opportunities found: {len(opportunities)}")
display(opportunities)

if len(opportunities):
    total_potential = opportunities["Est_Profit_Per_Truckload"].sum()
    print(f"\nEstimated profit across top {len(opportunities)} truckloads: Rs {total_potential:,.0f}")
    print("(Note: with real data this margin will typically be far smaller than synthetic data's")
    print(" injected anomalies suggest -- report your real measured numbers, not this placeholder.)")

## Step 8 — Multi-scale CPU vs GPU benchmark harness
Run the SAME pipeline at increasing data sizes and time it. This produces the scaling curve.

In [ ]:
IS_GPU_PASS = False   # set True when running with cudf.pandas loaded
OUTPUT_CSV = "bench_gpu.csv" if IS_GPU_PASS else "bench_cpu.csv"

def run_pipeline_timed(df):
    """Returns (clean_seconds, analyze_seconds, total_seconds) -- tracked separately
    so the Streamlit app's Acceleration tab can show a stage-by-stage breakdown,
    not just a single combined number."""
    t0 = time.time()
    c = clean_pipeline(df)
    t1 = time.time()
    a = analyze_pipeline(c)
    t2 = time.time()
    return (t1 - t0), (t2 - t1), (t2 - t0)

SCALES = [100_000, 500_000, 1_000_000, 2_000_000]
# Once confident it runs cleanly, extend toward your full dataset size, e.g.:
# SCALES = [100_000, 1_000_000, 5_000_000, 10_000_000, len(df_raw)]

bench_rows = []
for n in SCALES:
    n = min(n, len(df_raw))
    sub = df_raw.sample(n=n, random_state=1)
    try:
        clean_s, analyze_s, total_s = run_pipeline_timed(sub)
        bench_rows.append({
            "n_rows": n, "clean_seconds": clean_s, "analyze_seconds": analyze_s,
            "seconds": total_s, "pass": "gpu" if IS_GPU_PASS else "cpu",
        })
        print(f"scale={n:>10,}  clean={clean_s:6.2f}s  analyze={analyze_s:6.2f}s  total={total_s:6.2f}s")
    except MemoryError:
        bench_rows.append({"n_rows": n, "clean_seconds": None, "analyze_seconds": None, "seconds": None, "pass": "cpu_oom"})
        print(f"scale={n:>10,}  OOM -- this IS your acceleration proof, record it")
        break

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to {OUTPUT_CSV}")
bench_df

## Step 9 — Scaling comparison chart
Run this after you have both `bench_cpu.csv` and `bench_gpu.csv`.

In [ ]:
try:
    cpu_bench = pd.read_csv("bench_cpu.csv")
    gpu_bench = pd.read_csv("bench_gpu.csv")

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(cpu_bench["n_rows"], cpu_bench["seconds"], marker="o", color="#888888", label="CPU (pandas)")
    ax.plot(gpu_bench["n_rows"], gpu_bench["seconds"], marker="o", color="#76B900", label="GPU (cudf.pandas)")
    ax.set_xlabel("Dataset size (rows)")
    ax.set_ylabel("Pipeline runtime (seconds)")
    ax.set_title("Clean + analyze pipeline: CPU vs GPU at scale")
    ax.legend()
    ax.set_xscale("log")
    plt.tight_layout()
    plt.savefig("scaling_benchmark.png", dpi=150)
    plt.show()

    merged = cpu_bench.merge(gpu_bench, on="n_rows", suffixes=("_cpu", "_gpu"))
    merged["speedup"] = merged["seconds_cpu"] / merged["seconds_gpu"]
    print(merged[["n_rows", "seconds_cpu", "seconds_gpu", "speedup"]])
except FileNotFoundError as e:
    print(f"Missing benchmark file: {e}. Run Step 8 in both a CPU pass and a GPU pass first.")

## Step 10 — cuML risk clustering
Segments (market, commodity) pairs into risk/volatility tiers -- feeds a "which commodities need
closer daily monitoring" view in the decision layer.

In [ ]:
features = (
    df_analyzed.groupby(["Market", "Commodity"])
    .agg(mean_price=("Modal_Price", "mean"),
         volatility=("Modal_Price", "std"),
         mean_deviation=("Deviation_Pct", "mean"))
    .dropna()
    .reset_index()
)

X = StandardScaler().fit_transform(features[["mean_price", "volatility", "mean_deviation"]])

t0 = time.time()
km = GPU_KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
cluster_time = time.time() - t0
features["risk_cluster"] = km.labels_

print(f"Clustering time ({'cuML GPU' if CUML_AVAILABLE else 'sklearn CPU'}): {cluster_time:.3f}s")
print(features.groupby("risk_cluster")[["mean_price", "volatility", "mean_deviation"]].mean().round(1))
print("\nCluster sizes:")
print(features["risk_cluster"].value_counts())

## Step 11 — GPU-accelerated price forecast (XGBoost)
Predicts next-day modal price per (market, commodity) from lag features -- adds a forward-looking
"which markets are about to become opportunities" signal on top of the same-day ranking above.

In [ ]:
df_sorted = df_analyzed.sort_values(["Market", "Commodity", "Arrival_Date"])
df_sorted["Next_Day_Price"] = df_sorted.groupby(["Market", "Commodity"])["Modal_Price"].shift(-1)
df_sorted["Lag_1"] = df_sorted.groupby(["Market", "Commodity"])["Modal_Price"].shift(1)
df_sorted["Lag_2"] = df_sorted.groupby(["Market", "Commodity"])["Modal_Price"].shift(2)

feature_cols = ["Modal_Price", "Lag_1", "Lag_2", "Rolling_Mean_7d", "Z_Score", "Deviation_Pct"]
train_df = df_sorted.dropna(subset=["Next_Day_Price"] + feature_cols)

Xtr = train_df[feature_cols].values
ytr = train_df["Next_Day_Price"].values

try:
    model = xgb.XGBRegressor(n_estimators=100, max_depth=5, tree_method="hist", device="cuda")
    model.fit(Xtr, ytr)
    device_used = "GPU (cuda)"
except Exception:
    model = xgb.XGBRegressor(n_estimators=100, max_depth=5, tree_method="hist")
    model.fit(Xtr, ytr)
    device_used = "CPU (fallback)"

print(f"XGBoost trained on: {device_used}")
print("Feature importances:")
for f, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {f:<18} {imp:.3f}")

preds = model.predict(Xtr[:10])
print("\nSample predictions vs actuals:")
for p, a in zip(preds[:5], ytr[:5]):
    print(f"  predicted {p:.1f}  actual {a:.1f}")

## Step 11b — Export forecast for the Streamlit app
Produces `forecast.csv` (predicted next-day % change + confidence per commodity) and
`forecast_feature_importance.csv`.

In [ ]:
# Build a per-commodity next-day forecast summary from the trained XGBoost model
latest_by_commodity = (
    df_sorted.dropna(subset=feature_cols)
    .sort_values("Arrival_Date")
    .groupby("Commodity")
    .tail(1)
)

forecast_rows = []
for _, row in latest_by_commodity.iterrows():
    X_latest = row[feature_cols].values.reshape(1, -1)
    pred_price = model.predict(X_latest)[0]
    pred_change_pct = (pred_price - row["Modal_Price"]) / row["Modal_Price"] * 100
    forecast_rows.append({
        "Commodity": row["Commodity"],
        "Predicted_Change_Pct": round(float(pred_change_pct), 2),
        "Confidence": round(float(np.clip(1 - abs(row["Z_Score"]) / 10, 0.5, 0.95)), 2) if pd.notna(row["Z_Score"]) else 0.7,
    })

forecast_out = pd.DataFrame(forecast_rows)
forecast_out.to_csv("forecast.csv", index=False)

importance_out = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_,
}).sort_values("Importance", ascending=False)
importance_out.to_csv("forecast_feature_importance.csv", index=False)

print("Exported forecast.csv and forecast_feature_importance.csv")
display(forecast_out)
display(importance_out)

## Step 12 — Export results + BigQuery push

In [ ]:
opportunities.to_csv("top_opportunities.csv", index=False)
features.to_csv("risk_clusters.csv", index=False)
df_analyzed.to_parquet("full_analyzed_data.parquet", index=False)
print("Exported: top_opportunities.csv, risk_clusters.csv, full_analyzed_data.parquet")

# Push to BigQuery for the Looker Studio / Streamlit layers:
import pandas_gbq
pandas_gbq.to_gbq(opportunities, "agmarknet.top_opportunities", project_id=PROJECT_ID, if_exists="replace")
pandas_gbq.to_gbq(features, "agmarknet.risk_clusters", project_id=PROJECT_ID, if_exists="replace")

## Next steps
1. Run Step 8 as a CPU pass, then restart and run it as a GPU pass -- feed both CSVs into Step 9
2. Push `SCALES` in Step 8 up toward your real dataset's full size once you trust the pipeline
3. Swap in real Agmarknet data + real market coordinates as early as possible
4. Sanity-check `RS_PER_KM_PER_QUINTAL` against a freight quote
5. Feed `top_opportunities.csv` into the Streamlit app and BigQuery into Looker Studio
